# 第7回 練習問題 解答

本ノートには、メインノート `GCI2026S_第7回_特徴量エンジニアリング.ipynb` の各章末の練習問題に対する解答を載せています。


## ライブラリの読み込み

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

# 日本語フォントの設定(import するだけで matplotlib に日本語フォントが適用される)
try:
    import japanize_matplotlib  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'japanize-matplotlib'])
    import japanize_matplotlib  # noqa: F401

RANDOM_STATE = 42

df = sns.load_dataset('titanic')


## 共通の前処理(4章以降の練習問題で使用)

メインノートの2章〜5章で作った前処理・特徴量を `df_imp` にまとめて再現しておきます。練習問題4以降ではこの `df_imp` をベースに特徴量を組み立てます。


In [ ]:
df_imp = df.copy()

# 2章: 欠損補完と欠損フラグ
df_imp['age'] = df_imp['age'].fillna(df_imp['age'].median())
df_imp['embarked'] = df_imp['embarked'].fillna(df_imp['embarked'].mode()[0])
df_imp['fare'] = df_imp['fare'].fillna(df_imp['fare'].median())
df_imp['deck_missing'] = df['deck'].isna().astype(int)

# 3章: 数値変数の変換
df_imp['fare_log'] = np.log1p(df_imp['fare'])
df_imp['family_size'] = df_imp['sibsp'] + df_imp['parch'] + 1
df_imp['fare_per_person'] = df_imp['fare'] / df_imp['family_size']
df_imp['age_x_pclass'] = df_imp['age'] * df_imp['pclass']

# 5章: 集約・仮説ベース特徴量
df_imp['pclass_mean_fare'] = df_imp['pclass'].map(df_imp.groupby('pclass')['fare'].mean())
df_imp['fare_vs_pclass_mean'] = df_imp['fare'] / df_imp['pclass_mean_fare']
df_imp['is_child'] = (df_imp['age'] < 13).astype(int)
df_imp['is_alone'] = (df_imp['family_size'] == 1).astype(int)
df_imp['is_woman_or_child'] = ((df_imp['sex'] == 'female') | (df_imp['age'] < 13)).astype(int)

y = df_imp['survived']

# CV評価用のモデル(メインノートと同じ設定)
models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)),
    ]),
    'DecisionTree': DecisionTreeClassifier(random_state=RANDOM_STATE),
}


## 練習問題 1

**問題**: titanic データセットで、チケット等級 `pclass` と生存 `survived` の関係を、表または棒グラフで確認してみましょう。


In [ ]:
survival_by_pclass = df.groupby('pclass')['survived'].mean()
print(survival_by_pclass)

survival_by_pclass.plot(kind='bar', color='steelblue')
plt.ylabel('生存率')
plt.title('pclass ごとの生存率')
plt.show()


1等は約63%、2等は約47%、3等は約24%と、等級が低いほど生存率が下がっていることが分かります。等級が生存予測に強く効く特徴量だと予想できます。


## 練習問題 2

**問題**: `embarked` を最頻値で補完する代わりに、`'Unknown'` という新しいカテゴリで補完してみましょう。補完後の `embarked` の値の分布を確認してください。


In [ ]:
embarked_unk = df['embarked'].fillna('Unknown')
embarked_unk.value_counts(dropna=False)


欠損していた2件が `Unknown` カテゴリにまとまりました。最頻値補完では欠損2件が `S` に混ざってしまいますが、`Unknown` で補完すると「欠損していた」という情報を残せます。


## 練習問題 3

**問題**: `age` を対数変換した特徴量を作り、ヒストグラムで対数変換前後の分布を比較してみましょう。


In [ ]:
age_log = np.log1p(df_imp['age'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_imp['age'].hist(bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('age 対数変換前')
age_log.hist(bins=30, ax=axes[1], color='steelblue')
axes[1].set_title('age 対数変換後')
plt.tight_layout()
plt.show()


`age` はもともとそれほど偏った分布ではないため、対数変換しても劇的には形が変わりません。**対数変換が効くのは「右に長い裾を引く分布」**であり、`age` のような自然な分布には大きな効果がない、ということを確認できます。


## 練習問題 4

**問題**: `who` カラム(man/woman/child)を One-Hot Encoding して `X_ch4` に追加し、CVスコアがどう変わるか確認してみましょう。


In [ ]:
# メインノート4章と同じ X_ch4 を再現する
sex_ohe = pd.get_dummies(df_imp['sex'], prefix='sex').astype(int)
embarked_ohe = pd.get_dummies(df_imp['embarked'], prefix='embarked').astype(int)
class_ohe = pd.get_dummies(df_imp['class'], prefix='class').astype(int)

X_ch4 = pd.concat([
    df_imp[['pclass', 'sibsp', 'parch', 'fare_log', 'fare_per_person', 'family_size', 'age_x_pclass', 'deck_missing']],
    sex_ohe, embarked_ohe, class_ohe,
], axis=1)

# who を One-Hot Encoding して X_ch4 に追加
who_ohe = pd.get_dummies(df_imp['who'], prefix='who').astype(int)
X_ch4_with_who = pd.concat([X_ch4, who_ohe], axis=1)

print(f'X_ch4 の特徴量 {X_ch4.shape[1]}個')
print(f'X_ch4 + who_ohe の特徴量 {X_ch4_with_who.shape[1]}個、追加列: {list(who_ohe.columns)}')
X_ch4_with_who.head()


特徴量の状態が確認できたので、CVスコアを比較します。


In [ ]:
for name, model in models.items():
    score_base = cross_val_score(model, X_ch4, y, cv=5, scoring='accuracy').mean()
    score_with = cross_val_score(model, X_ch4_with_who, y, cv=5, scoring='accuracy').mean()
    diff = score_with - score_base
    print(f'{name}: X_ch4 = {score_base:.4f}, X_ch4 + who_ohe = {score_with:.4f}, Δ {diff:+.4f}')


`who` を加えると、`sex` だけでは表せなかった「子供かどうか」という情報が新たに加わります。スコアが上がるかどうかは、モデルが既に `age` から子供性を読み取れているかにも依存します。


## 練習問題 5

**問題**: `sex` を単位として `fare` の平均をとった集約特徴量を作り、各乗客にその値を割り当ててみましょう。


In [ ]:
sex_mean_fare = df_imp.groupby('sex')['fare'].mean()
print('sex ごとの平均運賃:')
print(sex_mean_fare)

df_imp['sex_mean_fare'] = df_imp['sex'].map(sex_mean_fare)
df_imp[['sex', 'fare', 'sex_mean_fare']].head(10)


**女性の平均運賃のほうが男性より高い** ことが分かります。`sex` という単位でデータを集計することで、性別ごとの傾向を1つの数値で表現できる特徴量を作れました。


## 練習問題 6

**問題**: `random_id` のカテゴリ数を 50 / 200 / 500 と変えて、NG版のCVスコアがどう変わるか確認してみましょう。


In [ ]:
# メインノート6章のリーケージ実験と同じ基本特徴量を使う
base_cols = ['pclass', 'age', 'fare', 'sibsp', 'parch']
df_lk = df_imp.copy()
model = models['LogisticRegression']

for n_cats in [50, 200, 500]:
    rng = np.random.RandomState(RANDOM_STATE)
    df_lk['random_id'] = rng.randint(0, n_cats, size=len(df_lk))
    tm = df_lk.groupby('random_id')['survived'].mean()
    df_lk['random_id_te'] = df_lk['random_id'].map(tm)
    X = df_lk[base_cols + ['random_id_te']]
    score = cross_val_score(model, X, y, cv=5, scoring='accuracy').mean()
    avg_per_cat = len(df_lk) / n_cats
    print(f'カテゴリ数 {n_cats:>3}、1カテゴリ平均 {avg_per_cat:>5.1f}人: CV accuracy = {score:.4f}')


カテゴリ数が増えるほど、つまり1カテゴリあたりのサンプル数が減るほど、見かけ上のCVスコアが高くなります。これは Target Encoding の値が「自分の `survived`」に近づくためで、リーケージの効果が強くなっているのです。


## 練習問題 7

**問題**: RFE で残す特徴量の数 `n_features_to_select` を 5, 10, 15, 20 と変えて、それぞれの場合のCVスコアを比較してみましょう。


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE

# メインノート5章と同じ X_ch5 を再現する
sex_ohe = pd.get_dummies(df_imp['sex'], prefix='sex').astype(int)
embarked_ohe = pd.get_dummies(df_imp['embarked'], prefix='embarked').astype(int)
class_ohe = pd.get_dummies(df_imp['class'], prefix='class').astype(int)

X_ch4 = pd.concat([
    df_imp[['pclass', 'sibsp', 'parch', 'fare_log', 'fare_per_person', 'family_size', 'age_x_pclass', 'deck_missing']],
    sex_ohe, embarked_ohe, class_ohe,
], axis=1)
X_ch5 = pd.concat([X_ch4, df_imp[['pclass_mean_fare', 'fare_vs_pclass_mean', 'is_child', 'is_alone', 'is_woman_or_child']]], axis=1)
print('全特徴量数:', X_ch5.shape[1])


In [ ]:
model = models['LogisticRegression']
results = []
for n in [5, 10, 15, 20]:
    rfe = RFE(estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100), n_features_to_select=min(n, X_ch5.shape[1]))
    rfe.fit(X_ch5, y)
    X_sel = X_ch5.iloc[:, rfe.support_]
    score = cross_val_score(model, X_sel, y, cv=5, scoring='accuracy').mean()
    results.append({'n_features': n, 'CV accuracy': score})

results_df = pd.DataFrame(results)
print(results_df)

results_df.plot(x='n_features', y='CV accuracy', marker='o')
plt.title('RFE で残す特徴量数とCVスコア')
plt.ylabel('CV accuracy')
plt.show()


残す特徴量数を増やしていくとCVスコアは伸びていきますが、**あるところでピークを迎えて頭打ち** になります。「多ければ多いほど良い」とは限らないことを確認できました。


謝辞：データセットの利用について

本教材では、データ分析・可視化の学習目的で、Seaborn の load_dataset 関数から取得できるサンプルデータセット titanic を使用しています。

取得元：seaborn-data
https://github.com/mwaskom/seaborn-data

seaborn-data は、Seaborn のサンプルコードやドキュメントで利用するためのデータ配布用リポジトリです。本教材では、出典を明記したうえで、教育目的の実習用データとして使用しています。